In [ ]:
import pytesseract
import re
from PIL import Image
from pathlib import Path
from collections import defaultdict

print(f"Tesseract Version: {pytesseract.get_tesseract_version()}")

raw_dict = Path('../../data/01_raw')
raw_dict.mkdir(parents=True, exist_ok=True)
ocr_dict = Path('../../data/02_ocr')
ocr_dict.mkdir(parents=True, exist_ok=True)


# OEM => OCR Engine Mode (0-3) 3 erkennt automatisch was es brauch
# PSM => Page Segmentation Mode (1-13)
    # 1 = vollautomatische seitenaalyse mit osd
    # 3 = vollautomatische seitenaalyse ohne osd (standardwert)
    # 4 = einzelne TExtplatze mit variablen Schriftgrößen
    # 6 = einzelnen, textblock
    # 11 = findet so viel text wie möglich (sparse text
custom_config = r'--oem 3 --psm 6'

# collect Data and group
documents = defaultdict(list)

for img_path in raw_dict.glob('*.jpg'):
    # ^(.*?) nimmt alles von anfang an als Gruppe 1 (basisname)
    # (\d+)$ nimmt alle ziffern am ende des namens als gruppe 2 (seitenzahl)
    match = re.match(r"^(.*?)(\d+)$", img_path.stem)

    if match:
        base_name = match.group(1)
        documents[base_name].append(img_path)
    else:
        print(f"Filename {img_path.name} does not match the expected pattern.")

# iterate group documents
for base_name, files in documents.items():
    # important: file sort (logic order)
    files.sort(key=lambda p: int(re.match(r"^(.*?)(\d+)$", p.stem).group(2)))

    doc_name = base_name.rstrip('_')
    out_file = ocr_dict / f"{doc_name}.txt"
    
    print(f"Verarbeite Dokument: {doc_name} ({len(files)} Seiten) -> {out_file.name}")

    with open(out_file, mode="w", encoding="utf-8") as f:
        
        for page_path in files:
            
            with Image.open(page_path) as img:
                text = pytesseract.image_to_string(img, lang="deu_frak", config=custom_config)

                # separator
                f.write(f"\n{'='*20}\n--- {page_path.name} ---\n{'='*20}\n\n")
                f.write(text)
                f.write("\n")

print("OCR done")
    

# Pandas Dataframe mit Bounding Boxes und COnfidence Score
#test = pytesseract.image_to_data(img, lang='deu', config=custom_config)
#print (test)


Tesseract Version: 5.5.2
Verarbeite Dokument: R_9346_I_1 (6 Seiten) -> R_9346_I_1.txt
Verarbeite Dokument: R_9346_I_2 (10 Seiten) -> R_9346_I_2.txt
OCR done


In [63]:
#pdf_bytes = pytesseract.image_to_pdf_or_hocr(img, lang='deu', config=custom_config, extension='pdf')

# 2. Daten als physische Datei speichern
#output_path = 'ausgabe_dokument.pdf'
#with open(output_path, 'wb') as f:
#    f.write(pdf_bytes)

#print(f"PDF erfolgreich unter '{output_path}' gespeichert.")

# Fiftyone Dataset

In [42]:
import fiftyone as fo
import pytesseract
from PIL import Image

dataset = fo.Dataset.from_dir(
    dataset_dir="../../data/1",
    dataset_type=fo.types.ImageDirectory,
    name="OCR_Test"
)

for sample in dataset:
    filepath = sample.filepath
    img = Image.open(filepath)
    img_width, img_height = img.size
    
    # Nutze image_to_data, um Bounding Boxes und Konfidenzwerte zu erhalten
    ocr_data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)
    
    detections = []
    n_boxes = len(ocr_data['text'])
    
    for i in range(n_boxes):
        text = ocr_data['text'][i].strip()
        conf = int(ocr_data['conf'][i])
        
        # Filtere leere Erkennungen und Artefakte (Konfidenz < 0 bedeutet oft kein Text)
        if not text or conf < 0:
            continue
            
        # Absolute Tesseract-Koordinaten in Pixeln
        x = ocr_data['left'][i]
        y = ocr_data['top'][i]
        w = ocr_data['width'][i]
        h = ocr_data['height'][i]
        
        # Relative FiftyOne-Koordinaten (0.0 bis 1.0)
        rel_x = x / img_width
        rel_y = y / img_height
        rel_w = w / img_width
        rel_h = h / img_height
        
        # Erstelle die Detection
        detection = fo.Detection(
            label=text,
            bounding_box=[rel_x, rel_y, rel_w, rel_h],
            confidence=conf / 100.0  # FiftyOne erwartet oft Konfidenz zwischen 0 und 1
        )
        detections.append(detection)
        
    # Speichere die Liste der Detections im Sample unter einem neuen Feldnamen
    sample["tesseract_ocr"] = fo.Detections(detections=detections)
    sample.save()

print("OCR-Daten erfolgreich in das Dataset geladen.")

# 3. Starte die FiftyOne App zur Visualisierung
session = fo.launch_app(dataset)
session.wait()

FiftyOneConfigError: MongoDB could not be installed on your system. Please define a `database_uri` in your `fiftyone.core.config.FiftyOneConfig` to connect to yourown MongoDB instance or cluster 